# Ngay 29 30 Capstone · Colab
Bật GPU nếu có. Notebook tự chọn CUDA, MPS hoặc CPU. Chấp nhận luật cuộc thi trước khi tải. Thực thi và nộp submission thủ công. Cấu hình ít epoch để thử quy trình.

Thêm KAGGLE_JSON vào Colab Secrets và cấp quyền cho notebook. Cài gói kaggle nếu runtime chưa có.

In [ ]:
from pathlib import Path
import os, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
def train_epoch(model, loader, optimizer):
    model.train()
    for x,y in loader:
        optimizer.zero_grad()
        loss=nn.functional.cross_entropy(model(x.to(device)),y.to(device))
        loss.backward()
        optimizer.step()
@torch.inference_mode()
def predict(model, loader):
    model.eval()
    return torch.cat([model(batch[0].to(device)).argmax(1).cpu() for batch in loader])
print('Thiết bị:', device)

In [ ]:
# Lưu JSON của Kaggle trong Colab Secrets dưới tên KAGGLE_JSON.
# Hàm chỉ được gọi khi INPUT_DIR không có dữ liệu và bạn cần tải từ Kaggle.
def download_kaggle(slug, competition=True):
    from google.colab import userdata
    from pathlib import Path
    import os, shutil, subprocess, zipfile
    credentials=Path.home()/'.kaggle/kaggle.json'
    os.environ['KAGGLE_CONFIG_DIR']=str(credentials.parent)
    credentials.parent.mkdir(mode=0o700, exist_ok=True)
    installed=not credentials.exists()
    if installed:
        descriptor=os.open(credentials,os.O_WRONLY|os.O_CREAT|os.O_EXCL,0o600)
        with os.fdopen(descriptor,'w') as handle:
            handle.write(userdata.get('KAGGLE_JSON'))
    try:
        credentials.chmod(0o600)
        cli=shutil.which('kaggle')
        if cli is None:
            raise RuntimeError('Cài Kaggle CLI trong runtime trước khi tải.')
        target=(Path('/content')/slug.rsplit('/',1)[-1]).resolve()
        target.mkdir(exist_ok=True)
        kind,opt=('competitions','-c') if competition else ('datasets','-d')
        subprocess.run([cli,kind,'download',opt,slug,'-p',str(target)],check=True)
        for file in target.glob('*.zip'):
            with zipfile.ZipFile(file) as bundle:
                if any(not (target/member.filename).resolve().is_relative_to(target) for member in bundle.infolist()):
                    raise ValueError('Tệp nén chứa đường dẫn không hợp lệ.')
                bundle.extractall(target)
        return target
    finally:
        if installed:
            credentials.unlink(missing_ok=True)

In [ ]:
ROOT=Path('/content')
INPUT_DIR=download_kaggle('house-prices-advanced-regression-techniques',competition=True)

In [ ]:
# Chọn đúng schema của cuộc thi trước khi chạy.
TARGET='SalePrice'
ID='Id'
train=pd.read_csv(INPUT_DIR/'train.csv')
test=pd.read_csv(INPUT_DIR/'test.csv')
columns=[c for c in train.select_dtypes('number') if c not in (ID,TARGET) and c in test.columns]
assert columns and ID in test and TARGET in train
split=np.random.default_rng(2026).permutation(len(train))
fit_ids,val_ids,holdout_ids=np.split(split,[int(.65*len(split)),int(.8*len(split))])
x=train[columns].to_numpy(np.float32)
target=train[TARGET].to_numpy(np.float32)

def fit_once(indices,seed,width):
    seed_all(seed)
    sample=x[indices]
    median=np.nanmedian(sample,axis=0)
    median=np.where(np.isfinite(median),median,0)
    clean=np.where(np.isnan(sample),median,sample)
    mu=clean.mean(0)
    sd=clean.std(0)
    sd=np.where(sd==0,1,sd)
    transform=lambda a: torch.tensor((np.where(np.isnan(a),median,a)-mu)/sd)
    y_mu,y_sd=target[indices].mean(),max(float(target[indices].std()),1.)
    architecture=nn.Linear(len(columns),1) if width==0 else nn.Sequential(nn.Linear(len(columns),width),nn.ReLU(),nn.Linear(width,1))
    model=architecture.to(device)
    opt=torch.optim.Adam(model.parameters(),lr=.01)
    loader=DataLoader(TensorDataset(transform(sample),torch.tensor((target[indices]-y_mu)/y_sd)),64,shuffle=True)
    for _ in range(14):
        model.train()
        for batch,truth in loader:
            opt.zero_grad()
            loss=nn.functional.mse_loss(model(batch.to(device)).squeeze(1),truth.to(device))
            loss.backward()
            opt.step()
    @torch.inference_mode()
    def inference(features):
        model.eval()
        return model(transform(features).to(device)).squeeze(1).cpu().numpy()*y_sd+y_mu
    return inference

seeds=(17,23,41)
mae=lambda a,b: float(np.abs(np.asarray(a)-b).mean())
summary=[dict(model=width,seed=seed,device=str(device),mae=mae(fit_once(fit_ids,seed,width)(x[val_ids]),target[val_ids]))
         for seed in seeds for width in (0,24)]
validation=pd.DataFrame(summary)
display(validation)
width=int(validation.groupby('model').mae.mean().idxmin())
combined=np.r_[fit_ids,val_ids]
holdout=np.mean([fit_once(combined,seed,width)(x[holdout_ids]) for seed in seeds],axis=0)
print('MAE holdout:',mae(holdout,target[holdout_ids]),'Baseline:',mae(np.median(target[combined]),target[holdout_ids]))
pred=np.mean([fit_once(split,seed,width)(test[columns].to_numpy(np.float32)) for seed in seeds],axis=0)
pd.DataFrame({ID:test[ID],TARGET:np.maximum(pred,0)}).to_csv(ROOT/'capstone_submission.csv',index=False)